# Analysis v2: does the price-per-m² ranking hold with the whole city?

The original analysis (June 2026, 1,901 Argenprop listings, 30 barrios with n ≥ 10) found:

1. a clear price-per-m² map with **Puerto Madero in its own tier**, no overlap with any other barrio;
2. location, not unit size, drives price per m²;
3. expensive overall ≠ expensive per m².

This notebook re-runs only the **main ranking** (finding 1) on the expanded dataset
(September 2026, 59,069 listings from Mercado Libre + Argenprop, 48 barrios) and compares
it with the original, using the same method: median of `price / covered_m2` per barrio,
barrios with at least 10 listings.

In [1]:
import subprocess, io
import pandas as pd

pd.set_option("display.width", 160)
MIN_N = 10   # same floor as the original analysis

v2 = pd.read_csv("../data/listings.csv")
# original dataset, straight from the commit that published it
v1 = pd.read_csv(io.BytesIO(subprocess.check_output(["git", "show", "e601433:data/listings.csv"])))
print(f"v1: {len(v1):,} listings, {v1.neighborhood.nunique()} barrios")
print(f"v2: {len(v2):,} listings, {v2.neighborhood.nunique()} barrios  (sources: {v2.source.value_counts().to_dict()})")

v1: 1,901 listings, 39 barrios
v2: 59,069 listings, 48 barrios  (sources: {'mercadolibre': 58999, 'argenprop': 70})


In [2]:
def ranking(df):
    g = (df.groupby("neighborhood")
           .agg(n=("price_per_m2", "size"),
                median_usd_m2=("price_per_m2", "median"),
                p25=("price_per_m2", lambda s: s.quantile(.25)),
                p75=("price_per_m2", lambda s: s.quantile(.75)))
           .round(0))
    g = g[g.n >= MIN_N].sort_values("median_usd_m2", ascending=False)
    g["rank"] = range(1, len(g) + 1)
    return g

r1, r2 = ranking(v1), ranking(v2)
print(f"barrios ranked: v1 = {len(r1)}, v2 = {len(r2)}")

barrios ranked: v1 = 30, v2 = 48


## 1. Ranking v2: median USD/m² by barrio, all 48

In [3]:
r2[["rank", "n", "median_usd_m2", "p25", "p75"]].astype(int)

,rank,n,median_usd_m2,p25,p75
neighborhood,,,,,
Puerto Madero,1,978,6190,5246,7756
Palermo,2,9568,3912,3121,5000
Núñez,3,3063,3879,3242,4626
Belgrano,4,5299,3699,3000,4654
Colegiales,5,1257,3375,2822,3967
Saavedra,6,1235,3310,2778,3886
Coghlan,7,450,3157,2744,3598
Villa Urquiza,8,2460,3112,2644,3636
Villa Devoto,9,1496,3110,2561,3788


## 2. Comparison with the original ranking

`rank_v1` is the barrio's position among the 30 barrios of the original analysis, `rank_v2`
among the 48 of this one. Because 18 barrios are new, a raw rank shift is not comparable, so
`rank_v1_in_v2` re-ranks the v2 medians **restricted to the 30 original barrios**; `shift` is
`rank_v1 - rank_v1_in_v2` (positive = moved up).

In [4]:
cmp = r2.join(r1[["rank", "n", "median_usd_m2"]].add_suffix("_v1"), how="left")
cmp = cmp.rename(columns={"rank": "rank_v2", "n": "n_v2", "median_usd_m2": "median_v2"})
old = cmp[cmp.rank_v1.notna()].sort_values("median_v2", ascending=False).copy()
old["rank_v1_in_v2"] = range(1, len(old) + 1)
cmp["rank_v1_in_v2"] = old["rank_v1_in_v2"]
cmp["shift"] = cmp["rank_v1"] - cmp["rank_v1_in_v2"]
cmp["median_change_%"] = ((cmp.median_v2 / cmp.median_usd_m2_v1 - 1) * 100).round(0)
cmp["new"] = cmp.rank_v1.isna()

cols = ["rank_v2", "n_v2", "median_v2", "rank_v1", "n_v1", "median_usd_m2_v1", "rank_v1_in_v2", "shift", "median_change_%"]
out = cmp[cols].copy()
for c in cols:
    out[c] = out[c].astype("Int64")
out.index = [f"{b}  *NEW*" if new else b for b, new in zip(cmp.index, cmp.new)]
out

,rank_v2,n_v2,median_v2,rank_v1,n_v1,median_usd_m2_v1,rank_v1_in_v2,shift,median_change_%
Puerto Madero,1,978,6190,1,12,5788,1,0,7
Palermo,2,9568,3912,3,246,3129,2,1,25
Núñez,3,3063,3879,5,49,3014,3,2,29
Belgrano,4,5299,3699,7,187,2889,4,3,28
Colegiales,5,1257,3375,8,45,2875,5,3,17
Saavedra,6,1235,3310,4,30,3070,6,-2,8
Coghlan *NEW*,7,450,3157,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
Villa Urquiza,8,2460,3112,6,122,2921,7,-1,7
Villa Devoto,9,1496,3110,2,79,3157,8,-6,-1
Chacarita,10,510,3096,10,11,2706,9,1,14


### Notable movers (|shift| ≥ 5 positions among the original 30) and new barrios

In [5]:
movers = cmp[cmp["shift"].abs() >= 5].sort_values("shift", ascending=False)
print("MOVED UP:")
for b, r in movers[movers["shift"] > 0].iterrows():
    print(f"  {b:20} v1 #{int(r.rank_v1):2} -> #{int(r.rank_v1_in_v2):2} among the same 30  (median {int(r.median_usd_m2_v1):,} -> {int(r.median_v2):,} USD/m², n {int(r.n_v1)} -> {int(r.n_v2):,})")
print("MOVED DOWN:")
for b, r in movers[movers["shift"] < 0].iterrows():
    print(f"  {b:20} v1 #{int(r.rank_v1):2} -> #{int(r.rank_v1_in_v2):2} among the same 30  (median {int(r.median_usd_m2_v1):,} -> {int(r.median_v2):,} USD/m², n {int(r.n_v1)} -> {int(r.n_v2):,})")
print("\nNEW BARRIOS (no or too few listings in v1), with their v2 rank out of 48:")
for b, r in cmp[cmp.new].iterrows():
    print(f"  #{int(r.rank_v2):2} {b:20} median {int(r.median_v2):,} USD/m²  n={int(r.n_v2):,}")
# barrios that were ranked in v1 but do not reach n>=10 in v2 (should be none)
print("\nDropped from the ranking:", sorted(set(r1.index) - set(r2.index)) or "none")

MOVED UP:
  San Telmo            v1 #28 -> #17 among the same 30  (median 1,667 -> 2,464 USD/m², n 11 -> 955)
  Villa Crespo         v1 #19 -> #11 among the same 30  (median 2,314 -> 2,924 USD/m², n 64 -> 2,259)
  Barracas             v1 #25 -> #20 among the same 30  (median 1,907 -> 2,353 USD/m², n 20 -> 943)
MOVED DOWN:
  Villa Luro           v1 #17 -> #22 among the same 30  (median 2,342 -> 2,292 USD/m², n 20 -> 907)
  Villa Devoto         v1 # 2 -> # 8 among the same 30  (median 3,157 -> 3,110 USD/m², n 79 -> 1,496)
  Mataderos            v1 #12 -> #23 among the same 30  (median 2,630 -> 2,287 USD/m², n 15 -> 560)

NEW BARRIOS (no or too few listings in v1), with their v2 rank out of 48:
  # 7 Coghlan              median 3,157 USD/m²  n=450
  #12 Villa Ortúzar        median 2,932 USD/m²  n=458
  #16 Parque Chas          median 2,587 USD/m²  n=182
  #19 Agronomía            median 2,478 USD/m²  n=104
  #27 Villa Real           median 2,291 USD/m²  n=117
  #29 La Paternal          me

## 3. Is Puerto Madero still in its own tier?

"Own tier" in the original report meant two things: nearly double the next barrio at the
median, and **no overlap** of its distribution with any other barrio's. Both are checked here,
the second by comparing Puerto Madero's 10th percentile with every other barrio's 90th.

In [6]:
for name, df, r in [("v1", v1, r1), ("v2", v2, r2)]:
    pm = df[df.neighborhood == "Puerto Madero"].price_per_m2
    second = r.index[1]
    gap = r.median_usd_m2.iloc[0] / r.median_usd_m2.iloc[1]
    others = df[df.neighborhood != "Puerto Madero"]
    p90 = others.groupby("neighborhood").price_per_m2.quantile(.9)
    p90 = p90[p90.index.isin(r.index)]
    overlap = (p90 > pm.quantile(.1)).sum()
    print(f"{name}: Puerto Madero median {int(pm.median()):,} (n={len(pm):,}) vs #2 {second} {int(r.median_usd_m2.iloc[1]):,}  -> {gap:.2f}x")
    print(f"     PM p10 = {int(pm.quantile(.1)):,}, PM p25 = {int(pm.quantile(.25)):,}; barrios whose p90 exceeds PM's p10: {overlap}"
          + ("" if overlap == 0 else f"  -> {', '.join(f'{b} ({int(v):,})' for b, v in p90[p90 > pm.quantile(.1)].sort_values(ascending=False).items())}"))
    print(f"     share of PM listings below the #2 barrio's median: {(pm < r.median_usd_m2.iloc[1]).mean():.0%}\n")

v1: Puerto Madero median 5,788 (n=12) vs #2 Villa Devoto 3,157  -> 1.83x
     PM p10 = 4,643, PM p25 = 5,146; barrios whose p90 exceeds PM's p10: 0
     share of PM listings below the #2 barrio's median: 0%

v2: Puerto Madero median 6,190 (n=978) vs #2 Palermo 3,912  -> 1.58x
     PM p10 = 4,590, PM p25 = 5,246; barrios whose p90 exceeds PM's p10: 6  -> Palermo (6,140), Núñez (6,025), Belgrano (6,000), Recoleta (5,000), Villa Devoto (4,681), Colegiales (4,642)
     share of PM listings below the #2 barrio's median: 3%



## 4. Finding 2 — location vs unit size

Original method: a scatter of each barrio's median unit size against its median USD/m² (n ≥ 10),
read visually as "no pattern", with Puerto Madero (largest units, highest USD/m²) as the counter-example
to a small-unit premium. Here the same barrio-level view gets a number (Spearman/Pearson, with and
without Puerto Madero), and — new, only possible with 59k listings — a listing-level check: how much
of the variation in USD/m² is explained by the barrio, how much by unit size, and what size does
*within* a barrio.

In [7]:
import numpy as np

def spearman(a, b):
    # Pearson correlation of ranks == Spearman, without needing scipy
    return a.rank().corr(b.rank())

def barrio_table(df):
    g = (df.groupby("neighborhood")
           .agg(n=("price", "size"), median_usd_m2=("price_per_m2", "median"),
                median_price=("price", "median"), median_m2=("covered_m2", "median"))
           .round(0))
    return g[g.n >= MIN_N]

b1, b2 = barrio_table(v1), barrio_table(v2)

print("Barrio level: correlation of median unit size vs median USD/m²")
for name, b in [("v1 (30 barrios)", b1), ("v2 (48 barrios)", b2)]:
    for label, bb in [("all", b), ("without Puerto Madero", b.drop("Puerto Madero"))]:
        sp = spearman(bb.median_m2, bb.median_usd_m2)
        pe = bb.median_m2.corr(bb.median_usd_m2)
        print(f"  {name:16} {label:24} Spearman {sp:+.2f}   Pearson {pe:+.2f}")
print("\nMedian unit size by barrio, v2 (top/bottom 5 by USD/m²):")
print(b2.sort_values("median_usd_m2", ascending=False)[["median_usd_m2", "median_m2"]].astype(int).iloc[list(range(5)) + list(range(-5, 0))].to_string())

Barrio level: correlation of median unit size vs median USD/m²
  v1 (30 barrios)  all                      Spearman +0.22   Pearson +0.69
  v1 (30 barrios)  without Puerto Madero    Spearman +0.13   Pearson +0.15
  v2 (48 barrios)  all                      Spearman +0.12   Pearson +0.45
  v2 (48 barrios)  without Puerto Madero    Spearman +0.06   Pearson -0.00

Median unit size by barrio, v2 (top/bottom 5 by USD/m²):
               median_usd_m2  median_m2
neighborhood                           
Puerto Madero           6190         92
Palermo                 3912         50
Núñez                   3879         51
Belgrano                3699         57
Colegiales              3375         57
La Boca                 1434         50
Constitución            1410         46
Nueva Pompeya           1136         59
Villa Lugano            1053         55
Villa Soldati            957         65


In [8]:
# Listing level (v2 only; v1 was too thin). Work in logs so the ratios are additive.
d = v2.copy()
d["ly"] = np.log(d.price_per_m2)
d["lm"] = np.log(d.covered_m2)

def r2_categorical(y, groups):
    # share of variance explained by group means
    within = y.groupby(groups).transform("mean")
    return 1 - ((y - within) ** 2).sum() / ((y - y.mean()) ** 2).sum()

r2_barrio = r2_categorical(d.ly, d.neighborhood)
r_size = d.lm.corr(d.ly)
r2_size = r_size ** 2
# size within barrio: demean both variables by barrio, then correlate
d["ly_w"] = d.ly - d.groupby("neighborhood").ly.transform("mean")
d["lm_w"] = d.lm - d.groupby("neighborhood").lm.transform("mean")
r_size_within = d.lm_w.corr(d.ly_w)
# both together: barrio dummies + log size (residual of within-barrio regression)
slope = np.polyfit(d.lm_w, d.ly_w, 1)[0]
resid = d.ly_w - slope * d.lm_w
r2_both = 1 - (resid ** 2).sum() / ((d.ly - d.ly.mean()) ** 2).sum()

print(f"Listing level, n = {len(d):,}  (log USD/m²)")
print(f"  R² barrio only            : {r2_barrio:.3f}")
print(f"  R² unit size only         : {r2_size:.3f}   (Pearson r = {r_size:+.3f}, Spearman = {spearman(d.covered_m2, d.price_per_m2):+.3f})")
print(f"  R² barrio + size          : {r2_both:.3f}   -> size adds {r2_both - r2_barrio:.3f} on top of barrio")
print(f"  size within barrio        : r = {r_size_within:+.3f}; elasticity {slope:+.3f}  (doubling m² changes USD/m² by {(2**slope - 1) * 100:+.0f}%)")
print()
print("Per-barrio Spearman(covered_m2, USD/m²), v2:")
wb = d.groupby("neighborhood").apply(lambda g: spearman(g.covered_m2, g.price_per_m2)).round(2)
print(f"  median across barrios {wb.median():+.2f}, range {wb.min():+.2f} .. {wb.max():+.2f}; positive in {(wb > 0).sum()} of {len(wb)} barrios")
print("  most negative:", wb.sort_values().head(5).to_dict())
print("  most positive:", wb.sort_values().tail(5).to_dict())
print()
print("USD/m² by size bucket, v2 (median), overall and inside Palermo:")
bins = [15, 35, 50, 70, 100, 150, 500]
d["size_bucket"] = pd.cut(d.covered_m2, bins)
tab = d.groupby("size_bucket", observed=True).price_per_m2.agg(n="size", median="median")
tab["Palermo"] = d[d.neighborhood == "Palermo"].groupby("size_bucket", observed=True).price_per_m2.median()
tab["Villa Lugano"] = d[d.neighborhood == "Villa Lugano"].groupby("size_bucket", observed=True).price_per_m2.median()
print(tab.round(0).astype("Int64").to_string())

Listing level, n = 59,069  (log USD/m²)
  R² barrio only            : 0.463
  R² unit size only         : 0.007   (Pearson r = +0.082, Spearman = +0.034)
  R² barrio + size          : 0.463   -> size adds 0.000 on top of barrio
  size within barrio        : r = -0.005; elasticity -0.003  (doubling m² changes USD/m² by -0%)

Per-barrio Spearman(covered_m2, USD/m²), v2:
  median across barrios -0.04, range -0.62 .. +0.35; positive in 18 of 48 barrios
  most negative: {'Nueva Pompeya': -0.62, 'San Nicolás': -0.52, 'Constitución': -0.5, 'San Telmo': -0.49, 'Monserrat': -0.47}
  most positive: {'Villa Riachuelo': 0.23, 'Villa Ortúzar': 0.26, 'Núñez': 0.27, 'Puerto Madero': 0.28, 'Villa Devoto': 0.35}

USD/m² by size bucket, v2 (median), overall and inside Palermo:
                 n  median  Palermo  Villa Lugano
size_bucket                                      
(15, 35]     11563    2917     3870          1903
(35, 50]     17022    2919     3854          1111
(50, 70]     12539    2750    

## 5. Finding 3 — total price vs price per m² rank the city differently

Original method: rank barrios (n ≥ 10) by median total price and by median USD/m²; a barrio far
apart on the two lists pays for size, not for a per-m² premium. Recoleta was the case (3rd by total
price, 13th by USD/m²). Same computation on v1 and v2; `divergence = rank_price - rank_usd_m2`
(positive = ranks higher by total price than by USD/m², i.e. large units; negative = the opposite).

In [9]:
def divergence(b):
    b = b.copy()
    b["rank_usd_m2"] = b.median_usd_m2.rank(ascending=False, method="min").astype(int)
    b["rank_price"] = b.median_price.rank(ascending=False, method="min").astype(int)
    b["divergence"] = b.rank_price - b.rank_usd_m2
    return b

d1, d2 = divergence(b1), divergence(b2)
print("Rank correlation between the two orderings (Spearman):")
print(f"  v1: {spearman(d1.rank_price, d1.rank_usd_m2):.2f}   v2: {spearman(d2.rank_price, d2.rank_usd_m2):.2f}")
print()
for name, dd, big in [("v1", d1, 4), ("v2", d2, 6)]:
    rec = dd.loc["Recoleta"]; bel = dd.loc["Belgrano"]
    print(f"{name}: Recoleta  #{int(rec.rank_price)} by total price, #{int(rec.rank_usd_m2)} by USD/m²  (median price {int(rec.median_price):,}, {int(rec.median_m2)} m², {int(rec.median_usd_m2):,} USD/m²)")
    print(f"{name}: Belgrano  #{int(bel.rank_price)} by total price, #{int(bel.rank_usd_m2)} by USD/m²  (median price {int(bel.median_price):,}, {int(bel.median_m2)} m²)")
    print(f"{name}: barrios with |divergence| >= {big}:")
    for b, r in dd[dd.divergence.abs() >= big].sort_values("divergence", ascending=False).iterrows():
        print(f"     {b:20} price #{int(r.rank_price):2} vs USD/m² #{int(r.rank_usd_m2):2}  ({int(r.divergence):+d})   {int(r.median_price):>9,} USD  {int(r.median_m2):3} m²  {int(r.median_usd_m2):,} USD/m²")
    print()

Rank correlation between the two orderings (Spearman):
  v1: 0.78   v2: 0.93

v1: Recoleta  #4 by total price, #13 by USD/m²  (median price 179,000, 70 m², 2,560 USD/m²)
v1: Belgrano  #3 by total price, #7 by USD/m²  (median price 185,000, 62 m²)
v1: barrios with |divergence| >= 4:
     Chacarita            price #26 vs USD/m² #10  (+16)      90,000 USD   33 m²  2,706 USD/m²
     Saavedra             price #14 vs USD/m² # 4  (+10)     127,801 USD   42 m²  3,070 USD/m²
     Retiro               price #23 vs USD/m² #14  (+9)     110,000 USD   53 m²  2,438 USD/m²
     Villa Urquiza        price #13 vs USD/m² # 6  (+7)     135,000 USD   49 m²  2,921 USD/m²
     Villa del Parque     price #16 vs USD/m² #11  (+5)     120,500 USD   50 m²  2,662 USD/m²
     San Cristóbal        price #28 vs USD/m² #24  (+4)      84,500 USD   50 m²  1,916 USD/m²
     Belgrano             price # 3 vs USD/m² # 7  (-4)     185,000 USD   62 m²  2,889 USD/m²
     Mataderos            price # 8 vs USD/m² #12  (-4)  

In [10]:
cols = ["n", "median_price", "median_m2", "median_usd_m2", "rank_price", "rank_usd_m2", "divergence"]
d2.sort_values("rank_price")[cols].astype(int)

,n,median_price,median_m2,median_usd_m2,rank_price,rank_usd_m2,divergence
neighborhood,,,,,,,
Puerto Madero,978,530000,92,6190,1,1,0
Belgrano,5299,215000,57,3699,2,4,-2
Recoleta,4912,198000,62,3037,3,11,-8
Palermo,9568,195000,50,3912,4,2,2
Núñez,3063,192000,51,3879,5,3,2
Villa Devoto,1496,184000,64,3110,6,9,-3
Colegiales,1257,183000,57,3375,7,5,2
Retiro,877,175000,60,2571,8,17,-9
Saavedra,1235,160650,48,3310,9,6,3


## Read-out

**Finding 2 (location, not size) — holds, and is now much stronger than the original could show.**

- Barrio level, same method as June: Spearman(median m², median USD/m²) = +0.12 across 48 barrios,
  +0.06 without Puerto Madero (v1: +0.22 / +0.13). Pearson +0.45 is entirely Puerto Madero (−0.00 without it).
  No relation, as before.
- Listing level (new, n = 59,069, log USD/m²): barrio explains **46%** of the variance; unit size explains
  **0.7%**, and **0.0% on top of barrio**. Within a barrio the size elasticity is −0.003: doubling the m²
  changes USD/m² by 0%. The small-unit premium hypothesis is rejected on average, more decisively than in June.
- Nuance the big sample adds: the *average* zero hides two opposite local patterns. In central and
  southern barrios small units do carry a premium (within-barrio Spearman: Nueva Pompeya −0.62, San Nicolás
  −0.52, Constitución −0.50, San Telmo −0.49, Monserrat −0.47), while in the north large units do
  (Villa Devoto +0.35, Puerto Madero +0.28, Núñez +0.27). And the overall size curve is U-shaped:
  15–100 m² sits flat at ~2,750–2,900 USD/m², above 100 m² it rises to 3,200–3,600 (Palermo: 3,750–3,870
  up to 5,110 for 150+ m²). "Size does not matter" is true on average; "a bigger unit never pays more
  per m²" is not.

**Finding 3 (total price and USD/m² rank the city differently) — holds, but the divergence is smaller than it looked.**

- Spearman between the two rankings: **0.78 in v1 → 0.93 in v2**. With 10–20 listings per barrio, part of
  the June divergence was sampling noise; with the whole city the two orderings mostly agree.
- Recoleta is still the clearest case: **#3 by median total price (198,000 USD) vs #11 by USD/m² (3,037)**,
  because its typical unit is 62 m² against Palermo's 50 (v1: #4 vs #13; the report said 3rd because its
  chart left Puerto Madero out). Belgrano's "gentler version" has essentially disappeared: #2 vs #4.
- Same pattern (large units, expensive in total, mid-table per m²): Retiro #8 vs #17 (60 m²), Flores #22 vs
  #32 (58 m²), Versalles #27 vs #38, Mataderos #21 vs #28. The mirror image (small units, cheap in total,
  pricey per m²): San Telmo #34 vs #21 (42 m²), Agronomía #33 vs #19 (44 m²), Parque Chas #23 vs #16 (42 m²).

**Honesty rule, as in section 2:** everything above compares orderings and within-dataset structure, which is
robust. Absolute levels (e.g. Recoleta 179,000 → 198,000 USD) mix a change of source (Mercado Libre vs
Argenprop) with three months, and this notebook cannot separate the two.